In [3]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



# 1 Define project files and haptic actions


BASE_DIR = Path.cwd()

SIMULATION_PATH = (
    BASE_DIR
    / "phase10_synthetic_learner_environment_rerun.csv"
)

POLICY_PATH = (
    BASE_DIR
    / "phase14_qlearning_policy_rerun.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "phase15_policy_comparison_rerun.csv"
)

EPISODE_OUTPUT_PATH = (
    BASE_DIR
    / "phase15_episode_results_rerun.csv"
)

PROFILE_OUTPUT_PATH = (
    BASE_DIR
    / "phase15_profile_comparison_rerun.csv"
)

REWARD_PLOT = (
    BASE_DIR
    / "phase15_policy_reward_comparison_rerun.png"
)

ACCURACY_PLOT = (
    BASE_DIR
    / "phase15_policy_accuracy_comparison_rerun.png"
)

DIAGNOSTIC_PATH = (
    BASE_DIR
    / "phase15_action_diagnostic_rerun.csv"
)

HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

HAPTIC_COST = {
    0: 0.00,
    1: 0.03,
    2: 0.08
}

print(
    "Simulation file exists:",
    SIMULATION_PATH.exists()
)

print(
    "Policy file exists:",
    POLICY_PATH.exists()
)

if not SIMULATION_PATH.exists():
    raise FileNotFoundError(
        f"Phase 10 file not found: {SIMULATION_PATH}"
    )

if not POLICY_PATH.exists():
    raise FileNotFoundError(
        f"Phase 14 policy not found: {POLICY_PATH}"
    )



# 2 Load the synthetic learner environment


simulation_df = pd.read_csv(
    SIMULATION_PATH
)

required_columns = [
    "Difficulty_D",
    "Cognitive_Effort_C",
    "D_level",
    "C_level",
    "Learner_Profile",
    "Learner_Ability",
    "Performance_P"
]

missing_columns = [
    column
    for column in required_columns
    if column not in simulation_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

print(
    "\nSimulation shape:",
    simulation_df.shape
)

print(
    "\nMissing simulation values:"
)

print(
    simulation_df[
        required_columns
    ]
    .isna()
    .sum()
)



# 3 Load the learned Phase 14 policy


policy_df = pd.read_csv(
    POLICY_PATH
)

policy = {}

for _, row in policy_df.iterrows():

    state = (
        int(
            row["D_level"]
        ),
        int(
            row["C_level"]
        ),
        int(
            row["P_level"]
        )
    )

    policy[
        state
    ] = int(
        row[
            "Haptic_Action"
        ]
    )

print(
    "\nLoaded policy states:",
    len(policy)
)

print(
    "Policy action counts:",
    Counter(
        policy.values()
    )
)

if len(policy) != 27:
    raise ValueError(
        "Expected a learned policy for all 27 states."
    )



# 4 Define learner performance levels


def performance_level(
    performance
):
    if performance < 1 / 3:
        return 0

    elif performance < 2 / 3:
        return 1

    else:
        return 2



# 5 Define the same Phase 13/14 environment


def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):
    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:
        return 0.00

    elif action == 1:
        return 0.15 * need

    elif action == 2:
        return 0.30 * need

    else:
        raise ValueError(
            "Action must be 0, 1, or 2."
        )


def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    return (
        1
        / (
            1
            + np.exp(-score)
        )
    )


def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    base_score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    effect = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    final_score = (
        base_score
        + effect
    )

    return (
        1
        / (
            1
            + np.exp(-final_score)
        )
    )


def calculate_reward(
    correct,
    p_before,
    p_after,
    action
):
    success_reward = (
        1.0
        if correct == 1
        else -0.5
    )

    improvement = (
        p_after
        - p_before
    )

    reward = (
        success_reward
        + 2.0 * improvement
        - HAPTIC_COST[
            action
        ]
    )

    return reward


def environment_step(
    row,
    performance,
    action,
    rng
):
    ability = float(
        row[
            "Learner_Ability"
        ]
    )

    difficulty = float(
        row[
            "Difficulty_D"
        ]
    )

    cognitive_effort = float(
        row[
            "Cognitive_Effort_C"
        ]
    )

    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    p_after = (
        probability_correct_with_haptics(
            ability=ability,
            difficulty=difficulty,
            cognitive_effort=cognitive_effort,
            performance=performance,
            action=action
        )
    )

    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    correct = rng.binomial(
        n=1,
        p=p_after
    )

    reward = calculate_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action
    )

    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )



# 6 Define fixed, random and learned policies


def select_action(
    policy_name,
    state,
    action_rng
):
    if policy_name == "Low":
        return 0

    elif policy_name == "Medium":
        return 1

    elif policy_name == "High":
        return 2

    elif policy_name == "Random":

        return int(
            action_rng.choice(
                [
                    0,
                    1,
                    2
                ]
            )
        )

    elif policy_name == "Q-learning":

        return int(
            policy[
                state
            ]
        )

    else:
        raise ValueError(
            f"Unknown policy: {policy_name}"
        )



# 7 Define the evaluation settings


POLICY_NAMES = [
    "Low",
    "Medium",
    "High",
    "Random",
    "Q-learning"
]

N_EPISODES = 300

STEPS_PER_EPISODE = 50

EVALUATION_SEED = 123

print(
    "\nEvaluation episodes:",
    N_EPISODES
)

print(
    "Steps per episode:",
    STEPS_PER_EPISODE
)

print(
    "Evaluation seed:",
    EVALUATION_SEED
)



# 8 Generate paired episode starting conditions


setup_rng = np.random.default_rng(
    EVALUATION_SEED
)

episode_start_indices = (
    setup_rng.integers(
        0,
        len(
            simulation_df
        ),
        size=N_EPISODES
    )
)

episode_seeds = (
    setup_rng.integers(
        0,
        2**32 - 1,
        size=N_EPISODES,
        dtype=np.uint32
    )
)

print(
    "\nPaired evaluation conditions created."
)



# 9 Evaluate one policy across the paired episodes


def evaluate_policy(
    policy_name
):
    episode_rows = []

    action_history = []

    for episode in range(
        N_EPISODES
    ):

        environment_rng = (
            np.random.default_rng(
                int(
                    episode_seeds[
                        episode
                    ]
                )
            )
        )

        # Separate RNG for Random-policy action choices
        action_rng = (
            np.random.default_rng(
                int(
                    episode_seeds[
                        episode
                    ]
                )
                + 1
            )
        )

        row = simulation_df.iloc[
            episode_start_indices[
                episode
            ]
        ]

        learner_profile = (
            row[
                "Learner_Profile"
            ]
        )

        performance = float(
            row[
                "Performance_P"
            ]
        )

        total_reward = 0.0

        total_correct = 0


        for step in range(
            STEPS_PER_EPISODE
        ):

            state = (
                int(
                    row[
                        "D_level"
                    ]
                ),

                int(
                    row[
                        "C_level"
                    ]
                ),

                performance_level(
                    performance
                )
            )


            action = select_action(
                policy_name=policy_name,
                state=state,
                action_rng=action_rng
            )

            action_history.append(
                action
            )


            (
                new_performance,
                reward,
                correct,
                p_before,
                p_after
            ) = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=environment_rng
            )


            total_reward += (
                reward
            )

            total_correct += (
                correct
            )


            # Next passage remains within the same learner profile
            learner_rows = (
                simulation_df[
                    simulation_df[
                        "Learner_Profile"
                    ]
                    == learner_profile
                ]
            )

            row = learner_rows.iloc[
                environment_rng.integers(
                    0,
                    len(
                        learner_rows
                    )
                )
            ]

            performance = (
                new_performance
            )


        episode_rows.append(
            {
                "Policy":
                    policy_name,

                "Episode":
                    episode + 1,

                "Learner_Profile":
                    learner_profile,

                "Mean_Reward":
                    total_reward
                    / STEPS_PER_EPISODE,

                "Accuracy":
                    total_correct
                    / STEPS_PER_EPISODE
            }
        )


    episode_df = pd.DataFrame(
        episode_rows
    )

    summary = {
        "Policy":
            policy_name,

        "Mean_Reward":
            episode_df[
                "Mean_Reward"
            ].mean(),

        "Reward_SD":
            episode_df[
                "Mean_Reward"
            ].std(
                ddof=1
            ),

        "Mean_Accuracy":
            episode_df[
                "Accuracy"
            ].mean(),

        "Accuracy_SD":
            episode_df[
                "Accuracy"
            ].std(
                ddof=1
            ),

        "Mean_Action":
            np.mean(
                action_history
            )
    }

    return (
        summary,
        episode_df,
        action_history
    )



# 10 Compare Q-learning with fixed and random baselines


summary_results = []

episode_results = []

q_actions = []

for policy_name in POLICY_NAMES:

    (
        summary,
        episode_df,
        action_history
    ) = evaluate_policy(
        policy_name
    )

    if policy_name == "Q-learning":
        q_actions = action_history

    summary_results.append(
        summary
    )

    episode_results.append(
        episode_df
    )


results_df = pd.DataFrame(
    summary_results
)

episode_results_df = pd.concat(
    episode_results,
    ignore_index=True
)

print(
    "\nPOLICY COMPARISON"
)

print(
    "-----------------"
)

print(
    results_df.round(4)
)


results_df.to_csv(
    OUTPUT_PATH,
    index=False
)

episode_results_df.to_csv(
    EPISODE_OUTPUT_PATH,
    index=False
)



# 11 Plot mean reward comparison

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df[
        "Policy"
    ],
    results_df[
        "Mean_Reward"
    ]
)

plt.xlabel(
    "Policy"
)

plt.ylabel(
    "Mean Reward per Step"
)

plt.title(
    "Haptic Policy Comparison: Mean Reward"
)

plt.tight_layout()

plt.savefig(
    REWARD_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 12 Plot mean accuracy comparison


plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df[
        "Policy"
    ],
    results_df[
        "Mean_Accuracy"
    ]
)

plt.xlabel(
    "Policy"
)

plt.ylabel(
    "Mean Simulated Accuracy"
)

plt.title(
    "Haptic Policy Comparison: Accuracy"
)

plt.tight_layout()

plt.savefig(
    ACCURACY_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 13 Summarise evaluation by learner profile


profile_results_df = (
    episode_results_df
    .groupby(
        [
            "Learner_Profile",
            "Policy"
        ],
        as_index=False
    )
    .agg(
        Mean_Reward=(
            "Mean_Reward",
            "mean"
        ),

        Reward_SD=(
            "Mean_Reward",
            "std"
        ),

        Mean_Accuracy=(
            "Accuracy",
            "mean"
        ),

        Accuracy_SD=(
            "Accuracy",
            "std"
        ),

        Episodes=(
            "Episode",
            "count"
        )
    )
)

print(
    "\nPOLICY RESULTS BY LEARNER PROFILE"
)

print(
    "---------------------------------"
)

print(
    profile_results_df.round(4)
)

profile_results_df.to_csv(
    PROFILE_OUTPUT_PATH,
    index=False
)



# 14 Examine the learned policy action distribution


policy_action_counts = Counter(
    policy.values()
)

print(
    "\nLearned policy action counts:"
)

for action in [
    0,
    1,
    2
]:

    count = (
        policy_action_counts
        .get(
            action,
            0
        )
    )

    print(
        HAPTIC_ACTIONS[
            action
        ],
        ":",
        count,
        "/",
        len(
            policy
        )
    )



# 15 Report Q-learning actions actually used in evaluation


action_counts = Counter(
    q_actions
)

print(
    "\nQ-learning actions during evaluation:"
)

for action in [
    0,
    1,
    2
]:

    count = (
        action_counts
        .get(
            action,
            0
        )
    )

    proportion = (
        count
        / len(
            q_actions
        )
    )

    print(
        HAPTIC_ACTIONS[
            action
        ],
        ":",
        count,
        "proportion =",
        round(
            proportion,
            3
        )
    )



# 16 Diagnose reward and accuracy by haptic action


diagnostic_results = []

for action in [
    0,
    1,
    2
]:

    rng = np.random.default_rng(
        123
    )

    rewards = []

    probability_changes = []

    correct_results = []

    for i in range(
        5000
    ):

        row = simulation_df.iloc[
            rng.integers(
                0,
                len(
                    simulation_df
                )
            )
        ]

        performance = float(
            row[
                "Performance_P"
            ]
        )

        (
            new_performance,
            reward,
            correct,
            p_before,
            p_after
        ) = environment_step(
            row=row,
            performance=performance,
            action=action,
            rng=rng
        )

        rewards.append(
            reward
        )

        probability_changes.append(
            p_after
            - p_before
        )

        correct_results.append(
            correct
        )


    diagnostic_results.append(
        {
            "Action":
                action,

            "Haptic_Level":
                HAPTIC_ACTIONS[
                    action
                ],

            "Mean_Reward":
                np.mean(
                    rewards
                ),

            "Mean_Probability_Change":
                np.mean(
                    probability_changes
                ),

            "Accuracy":
                np.mean(
                    correct_results
                )
        }
    )


diagnostic_df = pd.DataFrame(
    diagnostic_results
)

print(
    "\nREWARD DIAGNOSTIC BY ACTION"
)

print(
    "---------------------------"
)

print(
    diagnostic_df.round(
        4
    )
)

diagnostic_df.to_csv(
    DIAGNOSTIC_PATH,
    index=False
)



# 17 Check final Phase 15 outputs


print(
    "\nSaved policy comparison:",
    OUTPUT_PATH
)

print(
    "Saved episode-level results:",
    EPISODE_OUTPUT_PATH
)

print(
    "Saved profile comparison:",
    PROFILE_OUTPUT_PATH
)

print(
    "Saved action diagnostic:",
    DIAGNOSTIC_PATH
)

print(
    "Saved reward plot:",
    REWARD_PLOT
)

print(
    "Saved accuracy plot:",
    ACCURACY_PLOT
)



# 18 Confirm Phase 15 completion


print(
    "\nPHASE 15 COMPLETE"
)

print(
    "Policies evaluated:",
    POLICY_NAMES
)

print(
    "Episodes per policy:",
    N_EPISODES
)

print(
    "Steps per episode:",
    STEPS_PER_EPISODE
)

print(
    "Total Q-learning decisions analysed:",
    len(
        q_actions
    )
)

Simulation file exists: True
Policy file exists: True

Simulation shape: (2127, 37)

Missing simulation values:
Difficulty_D          0
Cognitive_Effort_C    0
D_level               0
C_level               0
Learner_Profile       0
Learner_Ability       0
Performance_P         0
dtype: int64

Loaded policy states: 27
Policy action counts: Counter({0: 10, 2: 9, 1: 8})

Evaluation episodes: 300
Steps per episode: 50
Evaluation seed: 123

Paired evaluation conditions created.

POLICY COMPARISON
-----------------
       Policy  Mean_Reward  Reward_SD  Mean_Accuracy  Accuracy_SD  Mean_Action
0         Low       0.2436     0.2648         0.4957       0.1766       0.0000
1      Medium       0.2635     0.2660         0.5105       0.1778       1.0000
2        High       0.2633     0.2632         0.5252       0.1766       2.0000
3      Random       0.2588     0.2668         0.5116       0.1785       0.9883
4  Q-learning       0.2630     0.2652         0.5120       0.1760       0.8987

POLICY RES